# What Distinguishes Excellent vs Terrible Entries?

We have:
- **84 Excellent entries** (36%) - Big gains, small drawdowns
- **71 Terrible entries** (31%) - Huge drawdowns

**Goal:** Find what's different about them so we can filter out the bad ones.

## Factors to Compare
1. **On-chain metrics at entry** - SOPR values, how deep the capitulation
2. **Price structure** - Distance from highs/lows, recent performance
3. **Moving averages** - Where is price relative to MAs
4. **Volatility** - Recent price volatility
5. **Time patterns** - Year, market cycle position
6. **Consecutive signals** - First signal vs repeated signals

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

print("Ready!")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()
df = df[df.index >= '2018-12-15']

# Load entry analysis from previous notebook
entry_df = pd.read_csv('../data/sopr_entry_analysis.csv', parse_dates=['entry_date'])

print(f"Price data: {len(df)} rows")
print(f"Entry analysis: {len(entry_df)} entries")
print(f"\nQuality distribution:")
print(entry_df['quality'].value_counts())

In [ ]:
# Add features to price dataframe
df['ma_20'] = df['price'].rolling(20).mean()
df['ma_50'] = df['price'].rolling(50).mean()
df['ma_100'] = df['price'].rolling(100).mean()
df['ma_200'] = df['price'].rolling(200).mean()

# Price relative to MAs
df['price_vs_ma20'] = (df['price'] / df['ma_20'] - 1) * 100
df['price_vs_ma50'] = (df['price'] / df['ma_50'] - 1) * 100
df['price_vs_ma200'] = (df['price'] / df['ma_200'] - 1) * 100

# Recent returns
df['return_7d'] = df['price'].pct_change(7) * 100
df['return_30d'] = df['price'].pct_change(30) * 100
df['return_90d'] = df['price'].pct_change(90) * 100

# Volatility (30-day rolling std of returns)
df['volatility_30d'] = df['price'].pct_change().rolling(30).std() * np.sqrt(365) * 100

# Distance from 52-week high/low
df['high_52w'] = df['price'].rolling(365).max()
df['low_52w'] = df['price'].rolling(365).min()
df['dist_from_high'] = (df['price'] / df['high_52w'] - 1) * 100
df['dist_from_low'] = (df['price'] / df['low_52w'] - 1) * 100

# SOPR features
df['sopr_ma7'] = df['sopr'].rolling(7).mean()
df['sopr_below_1_streak'] = (df['sopr'] < 1).astype(int)

print("Features created!")

In [ ]:
# Merge features with entry analysis
features_at_entry = []

for _, row in entry_df.iterrows():
    entry_date = row['entry_date']
    
    if entry_date in df.index:
        day_data = df.loc[entry_date]
        
        features_at_entry.append({
            'entry_date': entry_date,
            'quality': row['quality'],
            'max_drawdown': row['max_drawdown'],
            'max_gain': row['max_gain'],
            
            # Price
            'entry_price': row['entry_price'],
            
            # SOPR at entry
            'sopr': day_data['sopr'],
            'sopr_sth': day_data['sopr_sth'],
            'sopr_ma7': day_data['sopr_ma7'],
            
            # Price vs MAs
            'price_vs_ma20': day_data['price_vs_ma20'],
            'price_vs_ma50': day_data['price_vs_ma50'],
            'price_vs_ma200': day_data['price_vs_ma200'],
            
            # Recent returns
            'return_7d': day_data['return_7d'],
            'return_30d': day_data['return_30d'],
            'return_90d': day_data['return_90d'],
            
            # Volatility
            'volatility_30d': day_data['volatility_30d'],
            
            # Distance from extremes
            'dist_from_high': day_data['dist_from_high'],
            'dist_from_low': day_data['dist_from_low'],
            
            # Time features
            'year': entry_date.year,
            'month': entry_date.month,
        })

features_df = pd.DataFrame(features_at_entry)
print(f"Features extracted for {len(features_df)} entries")

In [ ]:
# Split into groups
excellent = features_df[features_df['quality'] == 'Excellent']
terrible = features_df[features_df['quality'] == 'Terrible']

print(f"Excellent entries: {len(excellent)}")
print(f"Terrible entries: {len(terrible)}")

---
## 1. Compare On-Chain Metrics at Entry

In [ ]:
def compare_feature(feature_name, excellent_df, terrible_df):
    """Compare a feature between excellent and terrible entries."""
    exc_vals = excellent_df[feature_name].dropna()
    ter_vals = terrible_df[feature_name].dropna()
    
    print(f"\n{feature_name}:")
    print(f"  {'Metric':<15} {'Excellent':>12} {'Terrible':>12} {'Diff':>12}")
    print(f"  {'-'*55}")
    print(f"  {'Median':<15} {exc_vals.median():>12.2f} {ter_vals.median():>12.2f} {exc_vals.median() - ter_vals.median():>+12.2f}")
    print(f"  {'Mean':<15} {exc_vals.mean():>12.2f} {ter_vals.mean():>12.2f} {exc_vals.mean() - ter_vals.mean():>+12.2f}")
    print(f"  {'Std':<15} {exc_vals.std():>12.2f} {ter_vals.std():>12.2f}")
    
    return exc_vals.median() - ter_vals.median()

In [ ]:
print("="*60)
print("ON-CHAIN METRICS AT ENTRY")
print("="*60)

compare_feature('sopr', excellent, terrible)
compare_feature('sopr_sth', excellent, terrible)
compare_feature('sopr_ma7', excellent, terrible)

In [ ]:
# Visualize SOPR distributions
fig = make_subplots(rows=1, cols=2, subplot_titles=['SOPR at Entry', 'STH SOPR at Entry'])

fig.add_trace(go.Histogram(x=excellent['sopr'], name='Excellent', marker_color='green', opacity=0.7, nbinsx=30), row=1, col=1)
fig.add_trace(go.Histogram(x=terrible['sopr'], name='Terrible', marker_color='red', opacity=0.7, nbinsx=30), row=1, col=1)

fig.add_trace(go.Histogram(x=excellent['sopr_sth'], name='Excellent', marker_color='green', opacity=0.7, nbinsx=30, showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=terrible['sopr_sth'], name='Terrible', marker_color='red', opacity=0.7, nbinsx=30, showlegend=False), row=1, col=2)

fig.add_vline(x=1, line_dash='dash', row=1, col=1)
fig.add_vline(x=1, line_dash='dash', row=1, col=2)

fig.update_layout(height=400, title_text='SOPR Values: Excellent vs Terrible Entries', barmode='overlay')
fig.show()

---
## 2. Compare Price Structure

In [ ]:
print("="*60)
print("PRICE VS MOVING AVERAGES")
print("="*60)

compare_feature('price_vs_ma20', excellent, terrible)
compare_feature('price_vs_ma50', excellent, terrible)
compare_feature('price_vs_ma200', excellent, terrible)

In [ ]:
# Visualize price vs MA200
fig = go.Figure()

fig.add_trace(go.Box(y=excellent['price_vs_ma200'], name='Excellent', marker_color='green'))
fig.add_trace(go.Box(y=terrible['price_vs_ma200'], name='Terrible', marker_color='red'))

fig.add_hline(y=0, line_dash='dash', line_color='black')

fig.update_layout(
    title='Price vs 200 MA at Entry (%)<br><sup>Positive = Above MA, Negative = Below MA</sup>',
    yaxis_title='% from 200 MA',
    height=500
)
fig.show()

# Calculate % above/below 200 MA
exc_above_200 = (excellent['price_vs_ma200'] > 0).mean() * 100
ter_above_200 = (terrible['price_vs_ma200'] > 0).mean() * 100

print(f"\n% of entries ABOVE 200 MA:")
print(f"  Excellent: {exc_above_200:.0f}%")
print(f"  Terrible: {ter_above_200:.0f}%")

---
## 3. Compare Recent Returns (Momentum)

In [ ]:
print("="*60)
print("RECENT RETURNS BEFORE ENTRY")
print("="*60)

compare_feature('return_7d', excellent, terrible)
compare_feature('return_30d', excellent, terrible)
compare_feature('return_90d', excellent, terrible)

In [ ]:
# Visualize recent returns
fig = make_subplots(rows=1, cols=3, subplot_titles=['7-Day Return', '30-Day Return', '90-Day Return'])

fig.add_trace(go.Box(y=excellent['return_7d'], name='Excellent', marker_color='green'), row=1, col=1)
fig.add_trace(go.Box(y=terrible['return_7d'], name='Terrible', marker_color='red'), row=1, col=1)

fig.add_trace(go.Box(y=excellent['return_30d'], name='Excellent', marker_color='green', showlegend=False), row=1, col=2)
fig.add_trace(go.Box(y=terrible['return_30d'], name='Terrible', marker_color='red', showlegend=False), row=1, col=2)

fig.add_trace(go.Box(y=excellent['return_90d'], name='Excellent', marker_color='green', showlegend=False), row=1, col=3)
fig.add_trace(go.Box(y=terrible['return_90d'], name='Terrible', marker_color='red', showlegend=False), row=1, col=3)

fig.add_hline(y=0, line_dash='dash', row=1, col=1)
fig.add_hline(y=0, line_dash='dash', row=1, col=2)
fig.add_hline(y=0, line_dash='dash', row=1, col=3)

fig.update_layout(height=400, title_text='Recent Returns Before Entry (%)')
fig.show()

---
## 4. Compare Distance from Highs/Lows

In [ ]:
print("="*60)
print("DISTANCE FROM 52-WEEK EXTREMES")
print("="*60)

compare_feature('dist_from_high', excellent, terrible)
compare_feature('dist_from_low', excellent, terrible)

In [ ]:
# Visualize
fig = make_subplots(rows=1, cols=2, subplot_titles=['Distance from 52w High', 'Distance from 52w Low'])

fig.add_trace(go.Box(y=excellent['dist_from_high'], name='Excellent', marker_color='green'), row=1, col=1)
fig.add_trace(go.Box(y=terrible['dist_from_high'], name='Terrible', marker_color='red'), row=1, col=1)

fig.add_trace(go.Box(y=excellent['dist_from_low'], name='Excellent', marker_color='green', showlegend=False), row=1, col=2)
fig.add_trace(go.Box(y=terrible['dist_from_low'], name='Terrible', marker_color='red', showlegend=False), row=1, col=2)

fig.update_layout(height=400, title_text='Distance from 52-Week Extremes (%)')
fig.show()

---
## 5. Compare Volatility

In [ ]:
print("="*60)
print("VOLATILITY")
print("="*60)

compare_feature('volatility_30d', excellent, terrible)

In [ ]:
# Visualize
fig = go.Figure()

fig.add_trace(go.Box(y=excellent['volatility_30d'], name='Excellent', marker_color='green'))
fig.add_trace(go.Box(y=terrible['volatility_30d'], name='Terrible', marker_color='red'))

fig.update_layout(
    title='30-Day Volatility at Entry (Annualized %)',
    yaxis_title='Volatility (%)',
    height=400
)
fig.show()

---
## 6. Compare Time Patterns

In [ ]:
print("="*60)
print("TIME PATTERNS")
print("="*60)

# Year breakdown
print("\nBy Year:")
year_summary = features_df.groupby(['year', 'quality']).size().unstack(fill_value=0)
print(year_summary)

In [ ]:
# Visualize by year
fig = go.Figure()

years = sorted(features_df['year'].unique())

exc_by_year = excellent.groupby('year').size()
ter_by_year = terrible.groupby('year').size()

fig.add_trace(go.Bar(x=exc_by_year.index, y=exc_by_year.values, name='Excellent', marker_color='green'))
fig.add_trace(go.Bar(x=ter_by_year.index, y=ter_by_year.values, name='Terrible', marker_color='red'))

fig.update_layout(
    title='Entry Quality by Year',
    xaxis_title='Year',
    yaxis_title='Count',
    barmode='group',
    height=400
)
fig.show()

# Calculate ratio
print("\nExcellent/Terrible Ratio by Year:")
for year in years:
    exc = len(excellent[excellent['year'] == year])
    ter = len(terrible[terrible['year'] == year])
    ratio = exc / ter if ter > 0 else float('inf')
    print(f"  {year}: {exc} excellent, {ter} terrible (ratio: {ratio:.2f})")

---
## 7. Scatter Plot: Find Separation

In [ ]:
# Create scatter plot matrix of key features
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['90d Return vs Dist from High',
                                   'Volatility vs 90d Return',
                                   'Price vs 200MA vs Dist from Low',
                                   'SOPR vs STH SOPR'])

# Plot 1: 90d return vs dist from high
fig.add_trace(go.Scatter(x=excellent['return_90d'], y=excellent['dist_from_high'],
                         mode='markers', marker=dict(color='green', size=8, opacity=0.6),
                         name='Excellent'), row=1, col=1)
fig.add_trace(go.Scatter(x=terrible['return_90d'], y=terrible['dist_from_high'],
                         mode='markers', marker=dict(color='red', size=8, opacity=0.6),
                         name='Terrible'), row=1, col=1)

# Plot 2: Volatility vs 90d return
fig.add_trace(go.Scatter(x=excellent['volatility_30d'], y=excellent['return_90d'],
                         mode='markers', marker=dict(color='green', size=8, opacity=0.6),
                         showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=terrible['volatility_30d'], y=terrible['return_90d'],
                         mode='markers', marker=dict(color='red', size=8, opacity=0.6),
                         showlegend=False), row=1, col=2)

# Plot 3: Price vs 200MA vs dist from low
fig.add_trace(go.Scatter(x=excellent['price_vs_ma200'], y=excellent['dist_from_low'],
                         mode='markers', marker=dict(color='green', size=8, opacity=0.6),
                         showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=terrible['price_vs_ma200'], y=terrible['dist_from_low'],
                         mode='markers', marker=dict(color='red', size=8, opacity=0.6),
                         showlegend=False), row=2, col=1)

# Plot 4: SOPR vs STH SOPR
fig.add_trace(go.Scatter(x=excellent['sopr'], y=excellent['sopr_sth'],
                         mode='markers', marker=dict(color='green', size=8, opacity=0.6),
                         showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(x=terrible['sopr'], y=terrible['sopr_sth'],
                         mode='markers', marker=dict(color='red', size=8, opacity=0.6),
                         showlegend=False), row=2, col=2)

fig.update_layout(height=800, title_text='Looking for Separation Between Excellent (Green) and Terrible (Red)')
fig.show()

---
## 8. Statistical Significance Tests

In [ ]:
from scipy import stats

def test_difference(feature_name):
    """Test if the difference between groups is statistically significant."""
    exc_vals = excellent[feature_name].dropna()
    ter_vals = terrible[feature_name].dropna()
    
    # T-test
    t_stat, p_value = stats.ttest_ind(exc_vals, ter_vals)
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt((exc_vals.std()**2 + ter_vals.std()**2) / 2)
    cohens_d = (exc_vals.mean() - ter_vals.mean()) / pooled_std if pooled_std > 0 else 0
    
    significant = "✓" if p_value < 0.05 else "✗"
    
    return {
        'feature': feature_name,
        'exc_median': exc_vals.median(),
        'ter_median': ter_vals.median(),
        'diff': exc_vals.median() - ter_vals.median(),
        'p_value': p_value,
        'cohens_d': cohens_d,
        'significant': significant
    }

# Test all features
features_to_test = ['sopr', 'sopr_sth', 'price_vs_ma20', 'price_vs_ma50', 'price_vs_ma200',
                    'return_7d', 'return_30d', 'return_90d', 'volatility_30d',
                    'dist_from_high', 'dist_from_low']

test_results = [test_difference(f) for f in features_to_test]
test_df = pd.DataFrame(test_results).sort_values('p_value')

print("\nSTATISTICAL SIGNIFICANCE TESTS")
print("="*90)
print(f"{'Feature':<20} {'Exc Med':>10} {'Ter Med':>10} {'Diff':>10} {'p-value':>10} {'Cohen\'s d':>10} {'Sig':>5}")
print("-"*90)
for _, row in test_df.iterrows():
    print(f"{row['feature']:<20} {row['exc_median']:>10.2f} {row['ter_median']:>10.2f} "
          f"{row['diff']:>+10.2f} {row['p_value']:>10.4f} {row['cohens_d']:>10.2f} {row['significant']:>5}")

---
## 9. Find Potential Filters

In [ ]:
# Test potential filter rules
def test_filter(name, condition):
    """Test a filter rule."""
    passed = features_df[condition]
    failed = features_df[~condition]
    
    exc_passed = len(passed[passed['quality'] == 'Excellent'])
    ter_passed = len(passed[passed['quality'] == 'Terrible'])
    exc_failed = len(failed[failed['quality'] == 'Excellent'])
    ter_failed = len(failed[failed['quality'] == 'Terrible'])
    
    # Quality rate in passed entries
    if len(passed) > 0:
        passed_quality = exc_passed / (exc_passed + ter_passed) if (exc_passed + ter_passed) > 0 else 0
    else:
        passed_quality = 0
    
    # How many excellent did we keep vs lose?
    exc_kept = exc_passed / (exc_passed + exc_failed) if (exc_passed + exc_failed) > 0 else 0
    ter_removed = ter_failed / (ter_passed + ter_failed) if (ter_passed + ter_failed) > 0 else 0
    
    return {
        'filter': name,
        'passed': len(passed),
        'exc_passed': exc_passed,
        'ter_passed': ter_passed,
        'passed_quality': passed_quality,
        'exc_kept': exc_kept,
        'ter_removed': ter_removed
    }

In [ ]:
# Test various filter rules
filters = [
    ('No filter (baseline)', pd.Series(True, index=features_df.index)),
    
    # Price vs MA filters
    ('Price > 200 MA', features_df['price_vs_ma200'] > 0),
    ('Price < 200 MA', features_df['price_vs_ma200'] < 0),
    ('Price > 50 MA', features_df['price_vs_ma50'] > 0),
    ('Price < 50 MA', features_df['price_vs_ma50'] < 0),
    
    # Recent return filters
    ('90d return > 0', features_df['return_90d'] > 0),
    ('90d return < 0', features_df['return_90d'] < 0),
    ('90d return > -20%', features_df['return_90d'] > -20),
    ('90d return < -30%', features_df['return_90d'] < -30),
    ('30d return < -10%', features_df['return_30d'] < -10),
    
    # Distance filters
    ('< 30% from high', features_df['dist_from_high'] > -30),
    ('> 30% from high', features_df['dist_from_high'] < -30),
    ('< 50% from high', features_df['dist_from_high'] > -50),
    ('> 50% from high', features_df['dist_from_high'] < -50),
    
    # Volatility filters
    ('Volatility < 80%', features_df['volatility_30d'] < 80),
    ('Volatility > 80%', features_df['volatility_30d'] > 80),
    ('Volatility < 100%', features_df['volatility_30d'] < 100),
    
    # SOPR filters
    ('SOPR < 0.98', features_df['sopr'] < 0.98),
    ('SOPR < 0.95', features_df['sopr'] < 0.95),
    ('STH SOPR < 0.95', features_df['sopr_sth'] < 0.95),
    
    # Combined filters
    ('90d ret > -30% AND vol < 100%', (features_df['return_90d'] > -30) & (features_df['volatility_30d'] < 100)),
    ('< 50% from high AND 90d ret > -40%', (features_df['dist_from_high'] > -50) & (features_df['return_90d'] > -40)),
]

filter_results = [test_filter(name, cond) for name, cond in filters]
filter_df = pd.DataFrame(filter_results).sort_values('passed_quality', ascending=False)

print("\nFILTER EFFECTIVENESS")
print("="*100)
print(f"{'Filter':<40} {'Passed':>8} {'Exc':>6} {'Ter':>6} {'Quality':>10} {'Exc Kept':>10} {'Ter Removed':>12}")
print("-"*100)
for _, row in filter_df.iterrows():
    print(f"{row['filter']:<40} {row['passed']:>8} {row['exc_passed']:>6} {row['ter_passed']:>6} "
          f"{row['passed_quality']*100:>9.0f}% {row['exc_kept']*100:>9.0f}% {row['ter_removed']*100:>11.0f}%")

---
## 10. Summary & Recommendations

In [ ]:
print("\n" + "="*70)
print("EXCELLENT vs TERRIBLE ENTRIES - KEY FINDINGS")
print("="*70)

# Find most significant differences
sig_features = test_df[test_df['p_value'] < 0.05].sort_values('p_value')

print(f"\n📊 STATISTICALLY SIGNIFICANT DIFFERENCES:")
if len(sig_features) > 0:
    for _, row in sig_features.head(5).iterrows():
        direction = "higher" if row['diff'] > 0 else "lower"
        print(f"   • {row['feature']}: Excellent entries have {direction} values")
        print(f"     (Excellent median: {row['exc_median']:.1f}, Terrible median: {row['ter_median']:.1f})")
else:
    print("   No statistically significant differences found!")

# Best filters
print(f"\n🎯 BEST FILTER CANDIDATES:")
best_filters = filter_df[(filter_df['passed'] >= 50) & (filter_df['filter'] != 'No filter (baseline)')].head(5)
for _, row in best_filters.iterrows():
    print(f"   • {row['filter']}")
    print(f"     Quality: {row['passed_quality']*100:.0f}% | Keeps {row['exc_kept']*100:.0f}% of Excellent | Removes {row['ter_removed']*100:.0f}% of Terrible")

print("\n" + "="*70)

In [ ]:
# Save analysis
features_df.to_csv('../data/sopr_entry_features.csv', index=False)
filter_df.to_csv('../data/sopr_filter_analysis.csv', index=False)
print("Saved to ../data/sopr_entry_features.csv and ../data/sopr_filter_analysis.csv")